In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import pandas as pd
import sys

# Get the absolute path to the target file's directory
scraper_dir = os.path.abspath('/Users/tylermartins/Documents/PitchSights/ps-betting/strategies/shots-on-target/models')
sys.path.append(scraper_dir)

# Now import from the file (without the .py extension)
import sot_model as sot


In [2]:
# Paths to reloaded files
processed_data_path = '/Users/tylermartins/Documents/PitchSights/ps-betting/strategies/shots-on-target/data/processed'

# Leagues:
# Premier-League - 9
# La-Liga - 12
# Serie-A - 11
# Ligue-1
# Bundesliga - 20

season = '2024-2025'
league = 'Premier-League'

df_cleaned = pd.read_csv(f"{processed_data_path}/features_{league}_{season}.csv")


# Step 3: Convert match_date to datetime and sort
df_cleaned['match_date'] = pd.to_datetime(df_cleaned['match_date'])
df_cleaned = df_cleaned.sort_values(by=['match_date', 'gameweek_number'])

# Step 4: Drop non-feature columns
columns_to_drop = [
    'match_date', 'player', 'line', 'bookmaker', 'under_odds',
       'team', 'position', 'age_decimal', 'minutes', 'goals', 'shots',
       'shots_on_target', 'xg', 'npxg', 'xg_assist', 'sca', 'gca',
       'miscontrols', 'dispossessed', 'passes_received',
       'progressive_passes_received', 'take_ons', 'take_ons_won',
       'take_ons_tackled', 'touches_att_3rd', 'touches_att_pen_area',
       'team_shots', 'team_shots_on_target', 'team_xG', 'home_team',
       'away_team', 'opp', 'opp_xG', 'team_score', 'opp_score',
       'team_possession', 'opp_possession', 'team_passing_accuracy',
       'opp_passing_accuracy', 'opp_shots_on_target', 'opp_shots',
       'team_saves', 'opp_saves', 'team_saves_faced', 'opp_saves_faced',
       'gameweek_number', 'shots_on_target_ma', 'goals_ma', 'shots_ma',
       'xg_ma', 'npxg_ma', 'xg_assist_ma', 'sca_ma', 'gca_ma',
       'touches_att_3rd_ma', 'touches_att_pen_area_ma', 'miscontrols_ma',
       'dispossessed_ma', 'passes_received_ma',
       'progressive_passes_received_ma', 'take_ons_won_ma',
       'take_ons_tackled_ma','model_confidence',
    'target_hit_line'  # this is the label, will be used in the model script
]
feature_cols = [col for col in df_cleaned.columns if col not in columns_to_drop]


In [3]:
# # Choose model type: "random_forest" or "xgboost"
# model_type = "xgboost"  # or "random_forest"

# # Prepare features and target
# X = df_cleaned[feature_cols]
# y = df_cleaned['target_hit_line']
# gameweeks = sorted(df_cleaned['gameweek_number'].unique())

# all_results = []

# for i in range(3, len(gameweeks)):
#     print("Training Week:", gameweeks[i])
#     train_weeks = gameweeks[:i]
#     test_week = gameweeks[i]

#     train_idx = df_cleaned['gameweek_number'].isin(train_weeks)
#     test_idx = df_cleaned['gameweek_number'] == test_week

#     X_train, y_train = X[train_idx], y[train_idx]
#     X_test, y_test = X[test_idx], y[test_idx]

#     # Scale features
#     scaler = StandardScaler()
#     X_train_scaled = scaler.fit_transform(X_train)
#     X_test_scaled = scaler.transform(X_test)

#     # Select model
#     if model_type == "random_forest":
#         model = RandomForestClassifier(n_estimators=100, random_state=42)
#     elif model_type == "xgboost":
#         model = xgb.XGBClassifier(
#             objective='binary:logistic',
#             eval_metric='logloss',
#             use_label_encoder=False,
#             n_estimators=100,
#             max_depth=3,
#             learning_rate=0.1,
#             random_state=42
#         )
#     else:
#         raise ValueError("Invalid model_type. Choose 'random_forest' or 'xgboost'.")

#     # Train and predict
#     model.fit(X_train_scaled, y_train)
#     y_pred = model.predict(X_test_scaled)
#     y_proba = model.predict_proba(X_test_scaled)[:, 1]

#     # Store results
#     results_df = df_cleaned.loc[test_idx, [
#         'player', 'match_date', 'gameweek_number', 'line', 'over_odds', 'shots_on_target'
#     ]].copy()
#     results_df['actual'] = y_test.values
#     results_df['predicted'] = y_pred
#     results_df['probability'] = y_proba

#     all_results.append(results_df)

# # Combine and display all results
# final_results = pd.concat(all_results, ignore_index=True)



In [4]:
# Choose model type
model_type = "xgboost"
gameweeks = sorted(df_cleaned['gameweek_number'].unique())

all_results = []
all_importance = []

for gw in gameweeks[3:]:
    print(f"Training and predicting for Gameweek {gw}")
    results, importance = sot.sot_model(df_cleaned,feature_cols, gw, model_type=model_type)
    all_results.append(results)
    all_importance.append(importance.set_index('feature'))

# Combine prediction results
final_results = pd.concat(all_results, ignore_index=True)

# Compute average feature importance across gameweeks
avg_importance_df = pd.concat(all_importance, axis=1).mean(axis=1).reset_index()
avg_importance_df.columns = ['feature', 'avg_importance']
avg_importance_df = avg_importance_df.sort_values(by='avg_importance', ascending=False)

# Display top 10 features
print("\n🔍 Top 10 Predictive Features (Shots on Target):")
print(avg_importance_df.head(10))


Training and predicting for Gameweek 7
Training and predicting for Gameweek 8


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:44] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:44] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:44] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training and predicting for Gameweek 9
Training and predicting for Gameweek 10


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:44] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:44] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training and predicting for Gameweek 11
Training and predicting for Gameweek 12


...

/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:47] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:48] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training and predicting for Gameweek 35
Training and predicting for Gameweek 36


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:48] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:48] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training and predicting for Gameweek 37
Training and predicting for Gameweek 38

🔍 Top 10 Predictive Features (Shots on Target):
                     feature  avg_importance
0                  over_odds        0.214867
29         implied_prob_over        0.098244
5                 minutes_ma        0.054676
1    team_shots_on_target_ma        0.043176
9                xg_per90_ma        0.039748
6          line_to_sot_ratio        0.037251
2   passes_received_per90_ma        0.032379
21            sot_team_share        0.029808
10             xg_team_share        0.027995
3              npxg_per90_ma        0.025091


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:48] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [5]:
ev_threshold = 0.08

final_results['implied_prob_over'] = 1 / final_results['over_odds']

final_results['ev'] = final_results['probability'] - final_results['implied_prob_over']


plus_ev = final_results[(final_results['ev'] > 0)&(final_results['predicted'] == 1)]
plus_ev_wins = final_results[(final_results['ev'] > 0)&(final_results['predicted'] == 1)&(final_results['actual'] == 1)]

ev_plus_co = final_results[(final_results['ev'] > ev_threshold)&(final_results['predicted'] == 1)]
ev_plus_co_wins = final_results[(final_results['ev'] > ev_threshold)&(final_results['predicted'] == 1)&(final_results['actual'] == 1)]


print(f'+EV Bets: {len(plus_ev)}')
print(f'+EV Wins: {len(plus_ev_wins)}')
print(f"+EV Win Rate: {len(plus_ev_wins) / len(plus_ev)}")

print(f"+EV Bets Avg. EV: {plus_ev['ev'].mean()}")
print(f"+EV Wins Avg. EV: {plus_ev_wins['ev'].mean()}")

print(f'EV + {ev_threshold*100}% Bets: {len(ev_plus_co)}')
print(f'EV + {ev_threshold*100}% Wins: {len(ev_plus_co_wins)}')
print(f"EV + {ev_threshold*100}% Win Rate: {len(ev_plus_co_wins) / len(ev_plus_co)}")

# print(f"+EV Bets Avg. EV: {plus_ev['ev'].mean()}")
# print(f"+EV Wins Avg. EV: {plus_ev_wins['ev'].mean()}")


+EV Bets: 830
+EV Wins: 482
+EV Win Rate: 0.5807228915662651
+EV Bets Avg. EV: 0.10490024613932528
+EV Wins Avg. EV: 0.10754075251560238
EV + 8.0% Bets: 403
EV + 8.0% Wins: 249
EV + 8.0% Win Rate: 0.6178660049627791


In [6]:
final_results[final_results['probability']>=0.6]

final_results[(final_results['ev'] > ev_threshold)&(final_results['predicted'] == 1)&(final_results['probability'] >= 0.6)]

,player,match_date,gameweek_number,bookmaker,line,over_odds,shots_on_target,actual,predicted,probability,implied_prob_over,ev
5,Bukayo Saka,2024-10-05,7,FanDuel,2.0,2.00,2,1,1,0.613777,0.500000,0.113777
6,Bukayo Saka,2024-10-05,7,BetRivers,0.5,1.30,2,1,1,0.897082,0.769231,0.127851
13,Declan Rice,2024-10-05,7,BetRivers,0.5,2.40,0,0,1,0.612051,0.416667,0.195385
14,Declan Rice,2024-10-05,7,FanDuel,1.0,1.87,0,0,1,0.707709,0.534759,0.172950
36,Gabriel Martinelli,2024-10-05,7,BetRivers,0.5,1.53,1,1,1,0.754362,0.653595,0.100767
...,...,...,...,...,...,...,...,...,...,...,...,...
27703,Brajan Gruda,2025-05-25,38,BetRivers,0.5,2.33,0,0,1,0.681148,0.429185,0.251963
27704,Brajan Gruda,2025-05-25,38,FanDuel,1.0,1.65,0,0,1,0.764737,0.606061,0.158677
27814,Beto,2025-05-25,38,BetRivers,0.5,1.82,0,0,1,0.640728,0.549451,0.091278
27988,Rasmus Højlund,2025-05-25,38,BetRivers,0.5,2.06,1,1,1,0.715382,0.485437,0.229945


In [10]:
ev_plus_co.to_csv(f'/Users/tylermartins/Documents/PitchSights/ps-betting/strategies/shots-on-target/outputs/sot_model_output_{league}_{season}.csv')

In [8]:
ev_plus_co[ev_plus_co['match_date'].astype(str).str.contains('2025-05-02')]

,player,match_date,gameweek_number,bookmaker,line,over_odds,shots_on_target,actual,predicted,probability,implied_prob_over,ev


In [9]:
df_cleaned[(df_cleaned['player'].str.contains('Cunha'))&(df_cleaned['match_date'].astype(str).str.contains('2025-05-10'))].to_csv(f'/Users/tylermartins/Documents/PitchSights/ps-betting/strategies/shots-on-target/data/processed/df_cleaned_test.csv')